# Logging and Error Handling

---

In this notebook, we will learn how to add **structured logging** and **error handling** to our ML API, so that we can monitor its behaviour in production.

We will cover:

- Why `print()` is not enough for production.
- Python's `logging` module.
- Structured logging with key-value fields.
- Logging predictions for monitoring and auditing.
- Graceful error handling in FastAPI
- What to log (and what not to log)

---

## 1. Why Logging Matters

When your API is running on a server (or inside a Docker container on Railway), you can't see what's happening. There's no terminal, no Jupyter output, no breakpoints. **Logs are your only window into the running system.**

| **Scenario** | **Without Logging** | **With Logging** |
| :--- | :--- | :--- |
| API returns wrong predictions | You have no idea why | You can see the exact input, the model's output, and the confidence |
| API crashes at 3am | Your find out when the client emails you | Your see the stack trace in `docker logs` |
| Slow responses | You don't know which requests are slow | You can see request duration for every call |
| Model loaded incorrectly | Silent failure | Startup log confirms model path and version |

---

## 2. Python's `logging` Module

`print()` writes to stdout, but it has no concept of severity, timestamps, or structured files. The `logging` module gives you all of this.

### Log Levels

| **Level** | **When to Use** | **Example** |
| :--- | :--- | :--- |
| `DEBUG` | Detailed diagnostic info. Off in production. | `logger.debug(f"Feature array shape: {X.shape}")` |
| `INFO` | Normal events you want to track. | `logger.info("Model loaded successfully.")` |
| `WARNING` | Something unexpected but not fatal. | `logger.warning("Low confidence prediction: 0.34")` |
| `ERROR` | Something failed, but the server is still running. | `logger.error(f"Prediction failed: {e}")` |
| `CRITICAL` | The server can't continue. | `logger.critical("Model file not found. Shutting down.")` |

### Basic Setup

In [ ]:
import logging 

# Create a logger for this module
logger = logging.getLogger(__name__)

# Configure the root logger to output to the console
logging.basicConfig(
    level=logging.INFO, # Set the logging level to INFO
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s", # Define the log message format
    datefmt="%Y-%m-%d %H:%M:%S", # Set the date format for log messages
)

Output:

```
2026-03-11 14:23:05 | INFO     | app.main | Model loaded successfully.
2026-03-11 14:23:07 | INFO     | app.main | Prediction: setosa (confidence=0.97)
2026-03-11 14:23:09 | WARNING  | app.main | Low confidence prediction: 0.34
```

Now every log line has a timestamp, severity level, and the module it came from - essential for debugging proctuion issues.

---

## 3. Logging Predictions

Every prediction your API makes should be logged. This creates an **audit trail** and enables monitoring over time.

In [ ]:
@app.post("/predict", response_model=IrisPrediction)
def predict(features: IrisFeatures) -> IrisPrediction:
    X = np.array([[features.sepal_length, features.sepal_width,
                   features.petal_length, features.petal_width]])

    pipeline = ml_model["pipeline"]
    prediction_id = int(pipeline.predict(X)[0])
    probabilities = pipeline.predict_proba(X)[0]
    confidence = float(probabilities[prediction_id])
    
    # Log the prediction details
    logger.info(
        "prediction_made | "
        f"input=[{features.sepal_length}, {features.sepal_width}, "
        f"{features.petal_length}, {features.petal_width}] | "
        f"prediction={TARGET_NAMES[prediction_id]} | "
        f"confidence={confidence:.4f}"
    )
    
    # Warn on low confidence
    if confidence < 0.6:
        logger.warning(
            f"low_confidence_prediction | confidence={confidence:.4f} | "
            f"predcition={TARGET_NAMES[prediction_id]}"
        )
        
    return IrisPrediction(
        prediction=TARGET_NAMES[prediction_id],
        prediction_id=prediction_id,
        probabilities={
            name: round(float(prob), 4)
            for name, prob in zip(TARGET_NAMES, probabilities)
        },
    )

### What This Gives You

- **Audit trail:** Every prediction is recorded with its input and output.
- **Low-confidence alerts:** A `WARNING` log when the model is uncertain - an early signal that something may be off.
- **Debugging:** If a client says *"Your model predicted X for this input"*, you can look up the exact log entry.

---

## 4. Error Handling in FastAPI

Unhandled exceptions crash the request and return a generic 500 error. We can do better:

In [ ]:
from fastapi import HTTPException

@app.post("/predict", response_model=IrisPrediction)
def predict(features: IrisFeatures) -> IrisPrediction:
    try:
        X = np.array([[features.sepal_length, features.sepal_width,
                       features.petal_length, features.petal_width]])

        pipeline = ml_model["pipeline"]
        prediction_id = int(pipeline.predict(X)[0])
        probabilities = pipeline.predict_proba(X)[0]

        logger.info(f"prediction_made | prediction={TARGET_NAMES[prediction_id]}")

        return IrisPrediction(
            prediction=TARGET_NAMES[prediction_id],
            prediction_id=prediction_id,
            probabilities={
                name: round(float(prob), 4)
                for name, prob in zip(TARGET_NAMES, probabilities)
            },
        )

    except Exception as e:
        logger.error(f"prediction_failed | error={e}", exc_info=True)
        raise HTTPException(
            status_code=500,
            detail="An internal error occurred during prediction. Please try again."
        )

|**Concept** | **Why** |
| :--- | :--- |
| `try/except` | Catches unexpected errors (model not loaded, bad array shape, etc.). |
| `exc_info=True` | Includes the full stack trace in the log - essential for debugging. |
| `HTTPException` | Returns a clean error to the client instead of a raw Python traceback. |
| Generic message to client | Don't expose internal details (`KeyError: pipeline`) - that's a security risk. Log the details, return a friendly message. |

---

## 5. Startup Logging

Log critical information when the server starts. This is the first thing you check when something goes wrong.

In [ ]:
@asynccontextmanager
async def lifespan(app: FastAPI):
    logger.info(f"startup | loading model from {MODEL_PATH}")
    ml_model["pipeline"] = load_pipeline()
    logger.info(
        f"startup | model loaded successfully | "
        f"type={type(ml_model['pipeline']).__name__} | "
        f"steps={[step[0] for step in ml_model['pipeline'].steps]}"
    )
    yield
    ml_model.clear()
    logger.info("shutdown | model cleared from memory")

Output:

```
2026-03-11 14:00:00 | INFO | app.main | startup | loading model from /app/models/iris_pipeline.joblib
2026-03-11 14:00:01 | INFO | app.main | startup | model loaded successfully | type=Pipeline | steps=['scaler', 'classifier']
```

If the model fails to load, the error log will tell you exactly what path it tried and what went wrong.

---

## 6. What to Log and What NOT to Log

### ✅ Do Log

| **What** | **Why** |
| :--- | :--- |
| Prediction inputs and outputs | Audit trail, drift detection, debugging |
| Confidence scores | Low-confidence alerts |
| Request duration | Performance monitoring |
| Errors with stack traces | Debugging |
| Model loading (path, version, type) | Verify correct model in production |
| Application startup/shutdown | Know when the service restarts |

### ❌ Do NOT Log

| **What** | **Why** |
| :--- | :--- |
| Passwords, API keys, tokens | Security: these should never appear in logs |
| Personally Identifiable Information (PII) | GDPR/privacy compliance |
| Full model weights | Too large, no value |
| Every DEBUG message in production | Noise. Ise `logging.INFO` as the production level |

---

## 7. Summary

| **Concept** | **Key Takeaway** |
| :--- | :--- |
| **`logging` module** | Use instead of `print()`. Gives you timestamps, levels, and module names. |
| **Log levels** | `DEBUG` for development, `INFO` for normal events, `WARNING` for concerning signals, `ERROR` for failures. |
| **Prediction logging** | Log every prediction's input, output, and confidence. This is your audit trail. |
| **Low-confidence warnings** | Flag uncertain predictions - an early signal of drift or out-of-distribution input. |
| **Error Handling** | Catch exceptions, log the full stack trace (`exc_info=True`), return a clean HTTP error to the client. |
| **Startup logging** | Log model path, type, and steps on startup. First thing to check when debugging. |

---

**Next:** [Data and Model Drift Concepts](./02_data_and_model_drift_concepts.ipynb): Understanding why model performance degrades and how to detect it.